# One Patient/Eager round — zones [0.05, 0.10, 0.20, 1.0]

Single-variable comparison with per-evaluation fit-time instrumentation.
`total fit work` = sum over evaluations of (mean fold fit time x 5): a
machine-noise-resistant measure of computation actually performed. The
shared-evaluation fit-time ratio isolates machine drift: the same params on
the same rows should cost the same if the machine is steady.

In [1]:
# --- Data pipeline: byte-for-byte replication of the prototype notebook ---
import time
import numpy as np
import pandas as pd

t0 = time.time()
trainBench = pd.read_csv(r"C:\FILES\Code\Benchmarking\train.csv")  # notebook: /home/dell/train.csv

SplitPoint = int(len(trainBench.index) * 0.8)
print("SplitPoint: ", SplitPoint)

df = trainBench

# int64 -> int32; object -> category -> numeric codes (Date included, as in the prototype)
Int64columns = df.select_dtypes(["int64"]).columns
df[Int64columns] = df[Int64columns].astype(np.int32)
cat_columns = df.select_dtypes(["object"]).columns
df[cat_columns] = df[cat_columns].astype("category")
df[cat_columns] = df[cat_columns].apply(lambda x: x.cat.codes)

# the five weather columns the prototype drops (note the dataset's own 'kM' typo)
df = df.drop("Max_Gust_SpeedKm_h", axis=1)
df = df.drop("CloudCover", axis=1)
df = df.drop("Max_VisibilityKm", axis=1)
df = df.drop("Min_VisibilitykM", axis=1)
df = df.drop("Mean_VisibilityKm", axis=1)

# positional 80/20 chronological split
trainBench, validBench = df.iloc[:SplitPoint, :], df.iloc[SplitPoint:, :]
print("Training Set shape", trainBench.shape)
print("Validation Set shape", validBench.shape)
del df

# feature mask: everything but the target (alphabetical order, incl. Date codes
# and NumberOfCustomers, exactly as the prototype had it)
mask = trainBench.columns.difference(["NumberOfSales"])
trainDataset_X = trainBench[mask]
trainDataset_y = trainBench["NumberOfSales"]
validBench_X = validBench[mask]
validBench_y = validBench["NumberOfSales"]
print("Feature Columns:")
print(list(mask))
del trainBench, validBench
print(f"Pipeline done in {time.time() - t0:.1f} s")

SplitPoint:  418416


C:\Users\Home\AppData\Local\Temp\ipykernel_19012\4255920482.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_columns = df.select_dtypes(["object"]).columns


Training Set shape (418416, 31)
Validation Set shape (104605, 31)
Feature Columns:
['AssortmentType', 'Date', 'Events', 'HasPromotions', 'IsHoliday', 'IsOpen', 'Max_Dew_PointC', 'Max_Humidity', 'Max_Sea_Level_PressurehPa', 'Max_TemperatureC', 'Max_Wind_SpeedKm_h', 'Mean_Dew_PointC', 'Mean_Humidity', 'Mean_Sea_Level_PressurehPa', 'Mean_TemperatureC', 'Mean_Wind_SpeedKm_h', 'Min_Dew_PointC', 'Min_Humidity', 'Min_Sea_Level_PressurehPa', 'Min_TemperatureC', 'NearestCompetitor', 'NumberOfCustomers', 'Precipitationmm', 'Region', 'Region_AreaKM2', 'Region_GDP', 'Region_PopulationK', 'StoreID', 'StoreType', 'WindDirDegrees']
Pipeline done in 4.1 s


In [2]:
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import TimeSeriesSplit
from pattern_search_cv import PatternSearchCV

X, y = trainDataset_X, trainDataset_y
N_features = X.shape[1]
tscv = TimeSeriesSplit(n_splits=5)
ZONES = [0.05, 0.10, 0.20, 1.0]

param_grid = {
    "max_features": [2, 3, 4],
    "n_estimators": list(range(10, 261, 10)),
    "max_depth": [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17],
}

def run_policy(policy):
    clf = ExtraTreesRegressor(n_estimators=240, max_features=max(1, N_features - 15),
                              max_depth=16, n_jobs=1, random_state=0)
    s = PatternSearchCV(clf, param_grid, scoring="neg_mean_absolute_error",
                        cv=tscv, n_jobs=-1, poll="opportunistic",
                        contraction=policy, data_zones=ZONES,
                        random_state=0, verbose=0)
    t0 = time.time()
    s.fit(X, y)
    wall = time.time() - t0
    res = s.cv_results_
    evals = []
    for p, sc, nr, ft in zip(res["params"], res["mean_test_score"],
                             res["n_resources"], res["mean_fit_time"]):
        key = (p["max_features"], p["n_estimators"], p["max_depth"], int(nr))
        evals.append({"key": key, "mae": -sc, "fit_work": float(ft) * 5})
    return {
        "policy": policy, "wall": wall,
        "n_fits": len(res["params"]),
        "equiv": float(np.sum(np.asarray(res["n_resources"]) / len(y))),
        "best": s.best_params_, "best_mae": -s.best_score_,
        "zones_used": sorted(set(int(v) for v in res["n_resources"])),
        "fit_work_total": sum(e["fit_work"] for e in evals),
        "evals": evals,
    }

def report(r):
    print(f"--- {r['policy'].upper()} ---")
    print(f"evaluations       : {r['n_fits']}")
    print(f"full-fit equiv    : {r['equiv']:.2f}")
    print(f"wall-clock        : {r['wall']:.1f} s")
    print(f"total fit work    : {r['fit_work_total']:.1f} s (sum of 5-fold fit times)")
    print(f"best              : ({r['best']['max_features']}, "
          f"{r['best']['n_estimators']}, {r['best']['max_depth']})  "
          f"MAE {r['best_mae']:.3f}")
    print(f"zones used (rows) : {r['zones_used']}")
    print()


In [3]:
P = run_policy("patient")
report(P)

--- PATIENT ---
evaluations       : 24
full-fit equiv    : 9.75
wall-clock        : 1552.3 s
total fit work    : 4422.9 s (sum of 5-fold fit times)
best              : (4, 230, 17)  MAE 815.373
zones used (rows) : [20921, 418416]



In [4]:
E = run_policy("eager")
report(E)

--- EAGER ---
evaluations       : 24
full-fit equiv    : 9.75
wall-clock        : 1769.4 s
total fit work    : 5091.3 s (sum of 5-fold fit times)
best              : (4, 230, 17)  MAE 815.373
zones used (rows) : [20921, 418416]



In [5]:
print("=" * 72)
print("P vs E - zones [0.05, 0.10, 0.20, 1.0], 26-value grid, MAE, seed 0")
print("=" * 72)
print(f"{'':22s}{'PATIENT':>12s}{'EAGER':>12s}")
print(f"{'evaluations':22s}{P['n_fits']:>12d}{E['n_fits']:>12d}")
print(f"{'full-fit equiv':22s}{P['equiv']:>12.2f}{E['equiv']:>12.2f}")
print(f"{'wall-clock (s)':22s}{P['wall']:>12.1f}{E['wall']:>12.1f}")
print(f"{'total fit work (s)':22s}{P['fit_work_total']:>12.1f}{E['fit_work_total']:>12.1f}")
print(f"{'best point':22s}"
      f"{str((P['best']['max_features'], P['best']['n_estimators'], P['best']['max_depth'])):>12s}"
      f"{str((E['best']['max_features'], E['best']['n_estimators'], E['best']['max_depth'])):>12s}")
print(f"{'best MAE':22s}{P['best_mae']:>12.3f}{E['best_mae']:>12.3f}")
print()
# machine-drift check: identical (params, rows) evaluated by BOTH runs
p_work = {e["key"]: e["fit_work"] for e in P["evals"]}
e_work = {e["key"]: e["fit_work"] for e in E["evals"]}
shared = sorted(set(p_work) & set(e_work))
if shared:
    ratios = [e_work[k] / p_work[k] for k in shared]
    print(f"shared evaluations: {len(shared)}  "
          f"(identical params AND rows in both runs)")
    print(f"eager/patient fit-time ratio on shared work: "
          f"median {sorted(ratios)[len(ratios)//2]:.3f}, "
          f"mean {sum(ratios)/len(ratios):.3f}")
    print("  (~1.0 = machine steady, differences are real work; "
          "<1.0 = machine was simply faster during the eager run)")
print()
print("unique-to-patient evals:", [k for k in p_work if k not in e_work])
print("unique-to-eager evals  :", [k for k in e_work if k not in p_work])

P vs E - zones [0.05, 0.10, 0.20, 1.0], 26-value grid, MAE, seed 0
                           PATIENT       EAGER
evaluations                     24          24
full-fit equiv                9.75        9.75
wall-clock (s)              1552.3      1769.4
total fit work (s)          4422.9      5091.3
best point            (4, 230, 17)(4, 230, 17)
best MAE                   815.373     815.373

shared evaluations: 24  (identical params AND rows in both runs)
eager/patient fit-time ratio on shared work: median 1.232, mean 1.306
  (~1.0 = machine steady, differences are real work; <1.0 = machine was simply faster during the eager run)

unique-to-patient evals: []
unique-to-eager evals  : []
